# TF IDF & CBOW

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re, sys, time
from google.colab import drive
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
pta_trunojoyo_path = "/content/drive/MyDrive/Crawling/pta_management.csv"
pta_trunojoyo = pd.read_csv(pta_trunojoyo_path)

pta_trunojoyo


,penulis,judul,pembimbing_pertama,pembimbing_kedua,abstrak_id,abstrak_en,prodi,url
0,SATIYAH,PENGARUH FAKTOR-FAKTOR PELATIHAN DAN PENGEMBAN...,"Dra. Hj. S. Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST.SE,M.MT","ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel...",ABSTRACT\r\n\r\nIn an effort to increase labor...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/080...
1,Faishal,ANALISIS PERSEPSI BRAND ASSOCIATION MENURUT PE...,Nurita Andriani,Yustina Chrismardani,Tujuan penelitian ini adalah untuk mengetahui ...,This study wanted to know the brand associatio...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/090...
2,Wahyu Kurniawan,PENGARUH GAYA KEPEMIMPINAN DEMOKRATIK TERHADAP...,"Dr. Dra. Hj. Iriani Ismail, MM","Dra. Hj. S. Anugrahini Irawati, MM",NaN,NaN,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/080...
3,Muhammad Zakaria Utomo,Pengukuran Website Quality Pada Situs Sistem A...,"Dr. Ir. Nurita Andriani, MM","Nirma Kurriwati, SP, M.Si",Aplikasi nyata pemanfaatan teknologi informasi...,Academic portal system in University of Trunoj...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/100...
4,Hendri Wahyudi Prayitno,PENGARUH KEPEMIMPINAN DAN KOMPENSASI TERHADAP ...,"Dra. Hj. S Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST,SE,.MT",Abstrak\r\nPenelitian ini menggunakan metode k...,Abstract\r\nThis research use quantitative met...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/080...
...,...,...,...,...,...,...,...,...
1026,Husnul Hotimah,Analisis Cost Volume Profit Untuk Menentukan T...,"Hj. Evaliati Amaniyah, S.E., M.S.M.",NaN,ABSTRAK\nPenelitian ini bertujuan untuk menget...,ABSTRACT\nThis study aims to determine the cal...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/160...
1027,Uswatun Hasanah,Pengaruh Pelatihan Dan Kompensasi Terhadap Pro...,"Dr. Raden Mas Mochammad Wispandono S.E ., MS",NaN,"ABSTRAK\nUswatun Hasanah, 160211100291, Pengar...","ABSTRACT\nUswatun Hasanah, 160211100291, The E...",Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/160...
1028,ACH FATHONI,PERAN SERVICE PERFORMANCE DAN CLIMATE ORGANIZA...,"YUDHI PRASETYA MADA, S.E., M.M.",NaN,ABSTRAK\nTujuan dari penelitian ini adalah unt...,ABSTRACK\n The purpose of this study...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/160...
1029,INTAN YULLIA NINGSIH,BAURAN PROMOSI PADA DEALER YAMAHA TRETAN MOTOR...,"DR. MOHAMMAD ARIEF, S.E., M.M.",NaN,ABSTRAK\nPenelitian ini bertujuan: (1) Untuk m...,ABSTRACK\nThis study aims: (1) To find out whe...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/160...


## **Mengecek Nilai yang null**


In [10]:
# Cek jumlah NaN di tiap kolom
print(pta_trunojoyo.isna().sum())


penulis                2
judul                  0
pembimbing_pertama     3
pembimbing_kedua      94
abstrak_id             5
abstrak_en             5
prodi                  0
url                    0
dtype: int64


In [11]:
# Hapus baris yang abstrak_en kosong
df = pta_trunojoyo.dropna(subset=['abstrak_id']).reset_index(drop=True)

# Cek lagi apakah masih ada yang kosong
print(df['abstrak_id'].isna().sum())


0


In [12]:
df

,penulis,judul,pembimbing_pertama,pembimbing_kedua,abstrak_id,abstrak_en,prodi,url
0,SATIYAH,PENGARUH FAKTOR-FAKTOR PELATIHAN DAN PENGEMBAN...,"Dra. Hj. S. Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST.SE,M.MT","ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel...",ABSTRACT\r\n\r\nIn an effort to increase labor...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/080...
1,Faishal,ANALISIS PERSEPSI BRAND ASSOCIATION MENURUT PE...,Nurita Andriani,Yustina Chrismardani,Tujuan penelitian ini adalah untuk mengetahui ...,This study wanted to know the brand associatio...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/090...
2,Muhammad Zakaria Utomo,Pengukuran Website Quality Pada Situs Sistem A...,"Dr. Ir. Nurita Andriani, MM","Nirma Kurriwati, SP, M.Si",Aplikasi nyata pemanfaatan teknologi informasi...,Academic portal system in University of Trunoj...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/100...
3,Hendri Wahyudi Prayitno,PENGARUH KEPEMIMPINAN DAN KOMPENSASI TERHADAP ...,"Dra. Hj. S Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST,SE,.MT",Abstrak\r\nPenelitian ini menggunakan metode k...,Abstract\r\nThis research use quantitative met...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/080...
4,Aththaariq,PENGARUH KOMPETENSI DOSEN TERHADAP KINERJA DOS...,"Dr.RM Moch Wispandono,.S.E,.MS","Dr. Muhammad Alkirom Wildan,S.E.,M.Si.","Abstrak\r\n\r\nAththaariq, Pengaruh Kompetensi...",Abstract\r\n\r\nThis study is aimed to analyze...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/080...
...,...,...,...,...,...,...,...,...
1021,Husnul Hotimah,Analisis Cost Volume Profit Untuk Menentukan T...,"Hj. Evaliati Amaniyah, S.E., M.S.M.",NaN,ABSTRAK\nPenelitian ini bertujuan untuk menget...,ABSTRACT\nThis study aims to determine the cal...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/160...
1022,Uswatun Hasanah,Pengaruh Pelatihan Dan Kompensasi Terhadap Pro...,"Dr. Raden Mas Mochammad Wispandono S.E ., MS",NaN,"ABSTRAK\nUswatun Hasanah, 160211100291, Pengar...","ABSTRACT\nUswatun Hasanah, 160211100291, The E...",Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/160...
1023,ACH FATHONI,PERAN SERVICE PERFORMANCE DAN CLIMATE ORGANIZA...,"YUDHI PRASETYA MADA, S.E., M.M.",NaN,ABSTRAK\nTujuan dari penelitian ini adalah unt...,ABSTRACK\n The purpose of this study...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/160...
1024,INTAN YULLIA NINGSIH,BAURAN PROMOSI PADA DEALER YAMAHA TRETAN MOTOR...,"DR. MOHAMMAD ARIEF, S.E., M.M.",NaN,ABSTRAK\nPenelitian ini bertujuan: (1) Untuk m...,ABSTRACK\nThis study aims: (1) To find out whe...,Manajemen,https://pta.trunojoyo.ac.id/welcome/detail/160...


## **Cleaning Data**

Kode tersebut melakukan pembersihan teks pada kolom abstrak_indo dengan mengubah semua huruf menjadi huruf kecil agar konsisten dan tidak membedakan antara huruf besar dan kecil. Selain itu, kode ini menghapus semua angka, tanda baca, dan simbol sehingga hanya tersisa huruf dan spasi, yang bertujuan menghilangkan karakter yang tidak relevan atau mengganggu proses analisis teks. Terakhir, pembersihan juga menghapus spasi berlebihan, termasuk spasi ganda atau spasi di awal dan akhir kalimat, agar teks menjadi lebih rapi dan mudah diproses. Proses ini penting untuk menyederhanakan dan menormalkan data teks sehingga siap digunakan dalam tahap analisis lebih lanjut seperti tokenisasi, stemming, atau klasifikasi.

In [13]:

import pandas as pd
import re

# Fungsi cleansing
def cleansing(text):
    text = str(text).lower()                         # ubah ke huruf kecil
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)        # hilangkan angka/simbol
    text = re.sub(r'\s+', ' ', text).strip()        # hilangkan spasi berlebih
    return text

# Terapkan cleansing
df['cleaned'] = df['abstrak_id'].apply(cleansing)

# Simpan ke CSV
df.to_csv('/content/drive/MyDrive/Crawling/manajemen_abstrak_cleaned.csv', index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan beberapa baris pertama
df[['abstrak_id','cleaned']].head()

Data berhasil disimpan ke CSV!



,abstrak_id,cleaned
0,"ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel...",abstrak satiyah pengaruh faktor faktor pelatih...
1,Tujuan penelitian ini adalah untuk mengetahui ...,tujuan penelitian ini adalah untuk mengetahui ...
2,Aplikasi nyata pemanfaatan teknologi informasi...,aplikasi nyata pemanfaatan teknologi informasi...
3,Abstrak\r\nPenelitian ini menggunakan metode k...,abstrak penelitian ini menggunakan metode kuan...
4,"Abstrak\r\n\r\nAththaariq, Pengaruh Kompetensi...",abstrak aththaariq pengaruh kompetensi dosen t...


In [ ]:
!pip install pyspellchecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 52.0 MB/s eta 0:00:00


## **Stopword Removal**
Kode ini digunakan untuk menghapus kata-kata umum (stopwords) dalam bahasa Indonesia dari teks yang sudah dibersihkan di kolom cleaned. Dengan menggunakan library Sastrawi, dibuat objek stopword remover yang akan menghilangkan kata-kata seperti “dan”, “di”, “yang”, dan kata umum lain yang biasanya tidak membawa makna penting dalam analisis teks.

Setelah proses penghilangan stopwords selesai, hasilnya disimpan dalam kolom baru bernama no_stopwords. Kemudian, data yang sudah diproses ini disimpan kembali ke file CSV baru manajemen_abstrak_no_stopwords.csv. Langkah ini penting agar teks menjadi lebih fokus pada kata-kata bermakna dan memudahkan analisis selanjutnya seperti klasifikasi atau clustering.

In [15]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 7.1 MB/s eta 0:00:00


In [16]:
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Buat stopword remover
factory = StopWordRemoverFactory()
stopword = factory.create_stop_word_remover()

# Terapkan stopword removal pada kolom 'cleaned'
df['no_stopwords'] = df['cleaned'].apply(stopword.remove)

# Simpan hasil ke CSV
df.to_csv('/content/drive/MyDrive/Crawling/manajemen_abstrak_no_stopwords.csv', index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan beberapa baris pertama
df[['cleaned','no_stopwords']].head()


Data berhasil disimpan ke CSV!



,cleaned,no_stopwords
0,abstrak satiyah pengaruh faktor faktor pelatih...,abstrak satiyah pengaruh faktor faktor pelatih...
1,tujuan penelitian ini adalah untuk mengetahui ...,tujuan penelitian mengetahui persepsi brand as...
2,aplikasi nyata pemanfaatan teknologi informasi...,aplikasi nyata pemanfaatan teknologi informasi...
3,abstrak penelitian ini menggunakan metode kuan...,abstrak penelitian menggunakan metode kuantita...
4,abstrak aththaariq pengaruh kompetensi dosen t...,abstrak aththaariq pengaruh kompetensi dosen k...


## **Stemming**
Kode ini melakukan proses stemming pada teks di kolom no_stopwords menggunakan library Sastrawi, yaitu mengubah kata-kata yang memiliki imbuhan atau variasi menjadi bentuk dasar atau kata dasarnya. Hasil stemming ini disimpan dalam kolom baru stemmed dan kemudian disimpan ke file CSV baru untuk digunakan dalam analisis selanjutnya. Proses stemming penting untuk menyederhanakan variasi kata sehingga model atau analisis teks dapat mengenali kata-kata dengan makna yang sama secara konsisten.

In [17]:
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Buat stemmer
stemmer = StemmerFactory().create_stemmer()

# Terapkan stemming pada kolom 'no_stopwords'
df['stemmed'] = df['no_stopwords'].apply(stemmer.stem)

# Simpan hasil ke CSV
df.to_csv('/content/drive/MyDrive/Crawling/manajemen_abstrak_stemmed.csv', index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan beberapa baris pertama
df[['no_stopwords','stemmed']].head()


Data berhasil disimpan ke CSV!



,no_stopwords,stemmed
0,abstrak satiyah pengaruh faktor faktor pelatih...,abstrak satiyah pengaruh faktor faktor latih k...
1,tujuan penelitian mengetahui persepsi brand as...,tuju teliti tahu persepsi brand association la...
2,aplikasi nyata pemanfaatan teknologi informasi...,aplikasi nyata manfaat teknologi informasi kom...
3,abstrak penelitian menggunakan metode kuantita...,abstrak teliti guna metode kuantitatif tekan u...
4,abstrak aththaariq pengaruh kompetensi dosen k...,abstrak aththaariq pengaruh kompetensi dosen k...


## **Tokenisasi**
Kode ini melakukan tokenisasi pada teks yang sudah melalui proses stemming di kolom stemmed. Tokenisasi adalah proses memecah kalimat atau teks menjadi potongan-potongan kata (token) yang lebih kecil, biasanya berdasarkan spasi. Fungsi tokenize yang sederhana di sini memecah setiap kalimat menjadi daftar kata-kata dengan menggunakan metode split(). Hasil tokenisasi disimpan dalam kolom baru tokens, yang berisi daftar kata untuk tiap baris teks. Setelah itu, data lengkap dengan token disimpan ke file CSV baru manajemen_abstrak_tokens.csv

In [18]:
import pandas as pd

# Fungsi tokenisasi sederhana
def tokenize(text):
    return text.split()

# Terapkan tokenisasi pada kolom 'stemmed'
df['tokens'] = df['stemmed'].apply(tokenize)

# Simpan hasil ke CSV
df.to_csv('/content/drive/MyDrive/Crawling/manajemen_abstrak_tokens.csv', index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan beberapa baris pertama
df[['stemmed','tokens']].head()


Data berhasil disimpan ke CSV!



,stemmed,tokens
0,abstrak satiyah pengaruh faktor faktor latih k...,"[abstrak, satiyah, pengaruh, faktor, faktor, l..."
1,tuju teliti tahu persepsi brand association la...,"[tuju, teliti, tahu, persepsi, brand, associat..."
2,aplikasi nyata manfaat teknologi informasi kom...,"[aplikasi, nyata, manfaat, teknologi, informas..."
3,abstrak teliti guna metode kuantitatif tekan u...,"[abstrak, teliti, guna, metode, kuantitatif, t..."
4,abstrak aththaariq pengaruh kompetensi dosen k...,"[abstrak, aththaariq, pengaruh, kompetensi, do..."


## **TF-IDF**

### **Ambil Corpus**

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Ambil corpus dari kolom stemmed
corpus = df['stemmed'].astype(str).tolist()

print("Jumlah dokumen dalam corpus:", len(corpus))
print("Contoh dokumen:", corpus[0][:200])


Jumlah dokumen dalam corpus: 1026
Contoh dokumen: abstrak satiyah pengaruh faktor faktor latih kembang produktivitas kerja dinas laut ikan bangkal bawah bimbing dra hj s anugrahini irawati mm helm buyung aulia s st se m mt upaya tingkat produktivitas


In [20]:
# Inisialisasi TF-IDF
vectorizer = TfidfVectorizer()

# Fit dan transform
X_tfidf = vectorizer.fit_transform(corpus)

print("Shape TF-IDF:", X_tfidf.shape)  # (jumlah dokumen, jumlah kata unik)


Shape TF-IDF: (1026, 6056)


In [21]:
# Ubah ke DataFrame biar mudah dibaca
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

# Simpan hasil ke CSV
tfidf_df.to_csv("/content/drive/MyDrive/Crawling/tfidf_hasil2.csv", index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan 5 baris pertama
tfidf_df.head()


Data berhasil disimpan ke CSV!



,aaa,aaaamanahsyariah,aar,abadi,abai,abalisis,abc,abcs,abd,abdul,...,zscore,zte,zuhri,zuhruf,zulfi,zulia,zuliana,zulkifli,zulpah,zyn
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
# Tampilkan 10 kata dengan bobot TF-IDF tertinggi di dokumen pertama
print(tfidf_df.iloc[0].sort_values(ascending=False).head(10))

produktivitas    0.411053
laut             0.331714
ikan             0.331714
latih            0.328843
dinas            0.234956
kembang          0.223807
pegawai          0.223024
faktor           0.222064
seleksi          0.165857
kerja            0.158086
Name: 0, dtype: float64


## **CBOW**

In [8]:
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 22.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.2
    Uninstalling scipy-1.16.2:
      Successfully uninstalled scipy-1.16.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatib

In [23]:
from gensim.models import Word2Vec

corpus = []
for col in df['stemmed']:
    word_list = col.split(" ")
    corpus.append(word_list)

print(corpus[0][:20])

['abstrak', 'satiyah', 'pengaruh', 'faktor', 'faktor', 'latih', 'kembang', 'produktivitas', 'kerja', 'dinas', 'laut', 'ikan', 'bangkal', 'bawah', 'bimbing', 'dra', 'hj', 's', 'anugrahini', 'irawati']


In [24]:
# Training Word2Vec
model = Word2Vec(
    corpus,
    vector_size=56,   # ukuran embedding
    window=5,
    min_count=1,
    sg=0   # CBOW
)


In [25]:
import numpy as np

class MeanEmbeddingVectorizer:
    def __init__(self, model):
        self.model = model
        self.vector_size = model.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.array([
            np.mean([self.model.wv[word] for word in words if word in self.model.wv]
                    or [np.zeros(self.vector_size)], axis=0)
            for words in X
        ])

    def fit_transform(self, X, y=None):
        return self.fit(X).transform(X)


In [26]:
mean_embedding_vectorizer = MeanEmbeddingVectorizer(model)
mean_embedded = mean_embedding_vectorizer.fit_transform(corpus)

# simpan ke DataFrame
df['array'] = list(mean_embedded)

# cek panjang embedding
df['embedding_length'] = df['array'].str.len()
print(df[['stemmed','embedding_length']].head())


                                             stemmed  embedding_length
0  abstrak satiyah pengaruh faktor faktor latih k...                56
1  tuju teliti tahu persepsi brand association la...                56
2  aplikasi nyata manfaat teknologi informasi kom...                56
3  abstrak teliti guna metode kuantitatif tekan u...                56
4  abstrak aththaariq pengaruh kompetensi dosen k...                56


In [27]:
num_features = len(df['array'].iloc[0])  # asumsi semua list punya panjang sama
columns = [f'f{i+1}' for i in range(num_features)]

# Inisialisasi dictionary untuk menampung data per kolom
data_dict = {col: [] for col in columns}

# Looping setiap baris di kolom 'embedding'
for embedding_list in df['array']:
    for i, value in enumerate(embedding_list):
        data_dict[f'f{i+1}'].append(value)

# Buat DataFrame dari dictionary
embedding_df = pd.DataFrame(data_dict)

print(embedding_df)

            f1        f2        f3        f4        f5        f6        f7  \
0    -0.656248  0.369275 -0.026647  0.123109  0.281716 -0.596587  0.687312   
1    -0.435555  0.647458  0.150943 -0.203099  0.088405 -0.610003  0.590858   
2    -0.439398  0.441261  0.075693  0.019303  0.209030 -0.509826  0.362906   
3    -0.571477  1.108523  0.093329  0.523282  0.613132 -0.224099  1.274294   
4    -0.533772  0.726877  0.111608  0.226298  0.310444 -0.423463  0.904037   
...        ...       ...       ...       ...       ...       ...       ...   
1021 -0.233128  0.595898 -0.115916  0.244533  0.355096 -0.033587  0.558574   
1022 -0.584110  0.777253 -0.083751  0.798399  0.987776 -0.073224  1.302692   
1023 -0.430942  0.941000  0.255556  0.403899  0.560167 -0.252602  0.981005   
1024 -0.724305  1.071965  0.234733 -0.100124 -0.033607 -0.495887  0.360477   
1025 -0.663332  0.340941 -0.003148  0.198481  0.421112 -0.571440  0.644504   

            f8        f9       f10  ...       f47       f48    

In [28]:
embedding_df['abstrak_id'] = df['abstrak_id'].values

In [29]:
embedding_df

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f48,f49,f50,f51,f52,f53,f54,f55,f56,abstrak_id
0,-0.656248,0.369275,-0.026647,0.123109,0.281716,-0.596587,0.687312,-0.443541,-0.245853,-0.392184,...,0.917156,-0.006421,0.387571,-0.033467,0.982716,-0.443007,-0.137661,0.380160,-0.188260,"ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel..."
1,-0.435555,0.647458,0.150943,-0.203099,0.088405,-0.610003,0.590858,-0.387401,-0.075915,-0.335614,...,0.102083,-0.029782,0.215439,-0.177409,0.796564,-0.539081,0.084412,0.455301,-0.038660,Tujuan penelitian ini adalah untuk mengetahui ...
2,-0.439398,0.441261,0.075693,0.019303,0.209030,-0.509826,0.362906,-0.340755,-0.131458,-0.340627,...,0.321093,0.014295,0.189817,-0.194312,0.729083,-0.337887,-0.061987,0.497773,-0.027140,Aplikasi nyata pemanfaatan teknologi informasi...
3,-0.571477,1.108523,0.093329,0.523282,0.613132,-0.224099,1.274294,-0.435111,-0.208434,-0.598882,...,1.082985,-0.236081,0.539385,0.078562,1.553989,-0.486495,-0.090007,0.381614,-0.166757,Abstrak\r\nPenelitian ini menggunakan metode k...
4,-0.533772,0.726877,0.111608,0.226298,0.310444,-0.423463,0.904037,-0.546627,-0.295800,-0.577074,...,0.639729,-0.217265,0.302807,-0.111081,1.149780,-0.505254,-0.048741,0.418941,-0.223363,"Abstrak\r\n\r\nAththaariq, Pengaruh Kompetensi..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1021,-0.233128,0.595898,-0.115916,0.244533,0.355096,-0.033587,0.558574,-0.729504,-0.182491,-0.536067,...,0.243420,-0.250277,-0.001709,0.104785,0.782429,-0.352461,0.213017,0.161028,0.233622,ABSTRAK\nPenelitian ini bertujuan untuk menget...
1022,-0.584110,0.777253,-0.083751,0.798399,0.987776,-0.073224,1.302692,-0.815880,-0.506612,-0.718611,...,1.377260,-0.182203,0.331992,0.335258,1.632860,-0.273281,-0.125063,0.256639,-0.214019,"ABSTRAK\nUswatun Hasanah, 160211100291, Pengar..."
1023,-0.430942,0.941000,0.255556,0.403899,0.560167,-0.252602,0.981005,-0.559020,-0.228040,-0.680139,...,0.470924,-0.204826,0.288336,0.022983,1.273404,-0.436177,0.054885,0.337636,0.066847,ABSTRAK\nTujuan dari penelitian ini adalah unt...
1024,-0.724305,1.071965,0.234733,-0.100124,-0.033607,-0.495887,0.360477,-0.537984,0.031398,-0.650937,...,0.873716,-0.155222,0.240221,-0.160307,0.961290,-0.319108,-0.056804,0.582850,-0.321127,ABSTRAK\nPenelitian ini bertujuan: (1) Untuk m...


In [30]:
embedding_df.shape

(1026, 57)

# **BERITA TF IDF & CBOW**

In [31]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re, sys, time
from google.colab import drive
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
berita_cnn_path = "/content/drive/MyDrive/Crawling/cnnindonesia_berita.csv"
berita_cnn = pd.read_csv(berita_cnn_path)

berita_cnn

,id_berita,judul_berita,isi_berita,kategori_berita
0,1,PM Qatar Desak Dunia Setop Pakai Standar Ganda...,Perdana MenteriQatarSheikh Mohammed bin Abdulr...,internasional
1,2,"Pengadilan Tak Mau Hukum Pengedar Miras, Akhir...",Sejarah pengambilalihan aset oleh pemerintahAm...,internasional
2,3,"Hasil Liga Inggris: Man City Bantai MU, Haalan...",Manchester Cityberhasil membantaiManchester Un...,olahraga
3,4,Danau Terbesar di Inggris Raya 'Sekarat' Akiba...,"Danau terbesar diInggris Raya, Lough Neagh, di...",internasional
4,5,Rizky Ridho Ungkap Niat Main di Luar Negeri: T...,"KaptenPersija Jakarta,Rizky Ridhomembuka dialo...",olahraga
...,...,...,...,...
995,996,Cara Cek Kesehatan Paru-Paru dan Tanda yang Ha...,Cara cekkesehatanparu-paru sering jadi pertany...,gaya-hidup
996,997,Komisi III DPR Buka Kans Dahulukan RUU Perampa...,Komisi III DPRsiap untuk mulai membahasRUU Per...,nasional
997,998,Kemenkeu Pastikan Rp200 T yang Ditebar ke Bank...,Kementerian Keuangan (Kemenkeu) memastikan uan...,ekonomi
998,999,"Sinopsis Gereja Setan, Terinspirasi Kisah Nyat...",Gereja Setanmerupakan film horor terbaru Indon...,hiburan


In [33]:
# Melihat daftar kategori unik
print(berita_cnn['kategori_berita'].unique())

# Menghitung jumlah kategori unik
print("Jumlah kategori:", berita_cnn['kategori_berita'].nunique())

# Kalau mau lihat jumlah data per kategori
print(berita_cnn['kategori_berita'].value_counts())

['internasional' 'olahraga' 'nasional' 'hiburan' 'gaya-hidup' 'ekonomi'
 'tv' 'otomotif' 'edukasi' 'teknologi']
Jumlah kategori: 10
kategori_berita
nasional         234
olahraga         171
ekonomi          169
internasional    141
tv               104
gaya-hidup        61
hiburan           44
teknologi         29
edukasi           25
otomotif          22
Name: count, dtype: int64


## **Mengecek Nilai yang null dan memebersihkannya**


In [34]:
# Cek jumlah NaN di tiap kolom
print(berita_cnn.isna().sum())

id_berita           0
judul_berita        0
isi_berita         39
kategori_berita     0
dtype: int64


In [35]:
# Hapus baris yang abstrak_en kosong
data = berita_cnn.dropna(subset=['isi_berita']).reset_index(drop=True)

# Cek lagi apakah masih ada yang kosong
print(data['isi_berita'].isna().sum())


0


In [36]:
data

,id_berita,judul_berita,isi_berita,kategori_berita
0,1,PM Qatar Desak Dunia Setop Pakai Standar Ganda...,Perdana MenteriQatarSheikh Mohammed bin Abdulr...,internasional
1,2,"Pengadilan Tak Mau Hukum Pengedar Miras, Akhir...",Sejarah pengambilalihan aset oleh pemerintahAm...,internasional
2,3,"Hasil Liga Inggris: Man City Bantai MU, Haalan...",Manchester Cityberhasil membantaiManchester Un...,olahraga
3,4,Danau Terbesar di Inggris Raya 'Sekarat' Akiba...,"Danau terbesar diInggris Raya, Lough Neagh, di...",internasional
4,5,Rizky Ridho Ungkap Niat Main di Luar Negeri: T...,"KaptenPersija Jakarta,Rizky Ridhomembuka dialo...",olahraga
...,...,...,...,...
956,996,Cara Cek Kesehatan Paru-Paru dan Tanda yang Ha...,Cara cekkesehatanparu-paru sering jadi pertany...,gaya-hidup
957,997,Komisi III DPR Buka Kans Dahulukan RUU Perampa...,Komisi III DPRsiap untuk mulai membahasRUU Per...,nasional
958,998,Kemenkeu Pastikan Rp200 T yang Ditebar ke Bank...,Kementerian Keuangan (Kemenkeu) memastikan uan...,ekonomi
959,999,"Sinopsis Gereja Setan, Terinspirasi Kisah Nyat...",Gereja Setanmerupakan film horor terbaru Indon...,hiburan


## **Cleaning Data**

In [38]:
import pandas as pd
import re

# Fungsi cleansing
def cleansing(text):
    text = str(text).lower()                         # ubah ke huruf kecil
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)        # hilangkan angka/simbol
    text = re.sub(r'\s+', ' ', text).strip()        # hilangkan spasi berlebih
    return text

# Terapkan cleansing
data['cleaned'] = data['isi_berita'].apply(cleansing)

# Simpan ke CSV
data.to_csv('/content/drive/MyDrive/Crawling/isi_berita_cleaned.csv', index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan beberapa baris pertama
data[['isi_berita','cleaned']].head()

Data berhasil disimpan ke CSV!



,isi_berita,cleaned
0,Perdana MenteriQatarSheikh Mohammed bin Abdulr...,perdana menteriqatarsheikh mohammed bin abdulr...
1,Sejarah pengambilalihan aset oleh pemerintahAm...,sejarah pengambilalihan aset oleh pemerintaham...
2,Manchester Cityberhasil membantaiManchester Un...,manchester cityberhasil membantaimanchester un...
3,"Danau terbesar diInggris Raya, Lough Neagh, di...",danau terbesar diinggris raya lough neagh dise...
4,"KaptenPersija Jakarta,Rizky Ridhomembuka dialo...",kaptenpersija jakarta rizky ridhomembuka dialo...


In [ ]:
!pip install pyspellchecker

## **Stopword Removal**

In [ ]:
!pip install Sastrawi

In [39]:
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Buat stopword remover
factory = StopWordRemoverFactory()
stopword = factory.create_stop_word_remover()

# Terapkan stopword removal pada kolom 'cleaned'
data['no_stopwords'] = data['cleaned'].apply(stopword.remove)

# Simpan hasil ke CSV
data.to_csv('/content/drive/MyDrive/Crawling/berita_cnn_no_stopwords.csv', index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan beberapa baris pertama
data[['cleaned','no_stopwords']].head()


Data berhasil disimpan ke CSV!



,cleaned,no_stopwords
0,perdana menteriqatarsheikh mohammed bin abdulr...,perdana menteriqatarsheikh mohammed bin abdulr...
1,sejarah pengambilalihan aset oleh pemerintaham...,sejarah pengambilalihan aset pemerintahamerika...
2,manchester cityberhasil membantaimanchester un...,manchester cityberhasil membantaimanchester un...
3,danau terbesar diinggris raya lough neagh dise...,danau terbesar diinggris raya lough neagh dise...
4,kaptenpersija jakarta rizky ridhomembuka dialo...,kaptenpersija jakarta rizky ridhomembuka dialo...


## **Stemming**

In [40]:
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Buat stemmer
stemmer = StemmerFactory().create_stemmer()

# Terapkan stemming pada kolom 'no_stopwords'
data['stemmed'] = data['no_stopwords'].apply(stemmer.stem)

# Simpan hasil ke CSV
data.to_csv('/content/drive/MyDrive/Crawling/berita_cnn_stemmed.csv', index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan beberapa baris pertama
data[['no_stopwords','stemmed']].head()


Data berhasil disimpan ke CSV!



,no_stopwords,stemmed
0,perdana menteriqatarsheikh mohammed bin abdulr...,perdana menteriqatarsheikh mohammed bin abdulr...
1,sejarah pengambilalihan aset pemerintahamerika...,sejarah pengambilalihan aset pemerintahamerika...
2,manchester cityberhasil membantaimanchester un...,manchester cityberhasil membantaimanchester un...
3,danau terbesar diinggris raya lough neagh dise...,danau besar inggris raya lough neagh sebut ala...
4,kaptenpersija jakarta rizky ridhomembuka dialo...,kaptenpersija jakarta rizky ridhomembuka dialo...


## **Tokenisasi**

In [41]:
import pandas as pd

# Fungsi tokenisasi sederhana
def tokenize(text):
    return text.split()

# Terapkan tokenisasi pada kolom 'stemmed'
data['tokens'] = data['stemmed'].apply(tokenize)

# Simpan hasil ke CSV
data.to_csv('/content/drive/MyDrive/Crawling/berita_cnn_tokens.csv', index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan beberapa baris pertama
data[['stemmed','tokens']].head()


Data berhasil disimpan ke CSV!



,stemmed,tokens
0,perdana menteriqatarsheikh mohammed bin abdulr...,"[perdana, menteriqatarsheikh, mohammed, bin, a..."
1,sejarah pengambilalihan aset pemerintahamerika...,"[sejarah, pengambilalihan, aset, pemerintahame..."
2,manchester cityberhasil membantaimanchester un...,"[manchester, cityberhasil, membantaimanchester..."
3,danau besar inggris raya lough neagh sebut ala...,"[danau, besar, inggris, raya, lough, neagh, se..."
4,kaptenpersija jakarta rizky ridhomembuka dialo...,"[kaptenpersija, jakarta, rizky, ridhomembuka, ..."


## **TF-IDF**

### **Ambil Corpus**

In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Ambil corpus dari kolom stemmed
corpus = data['stemmed'].astype(str).tolist()

print("Jumlah dokumen dalam corpus:", len(corpus))
print("Contoh dokumen:", corpus[0][:200])

Jumlah dokumen dalam corpus: 961
Contoh dokumen: perdana menteriqatarsheikh mohammed bin abdulrahman al thani desak dunia internasional henti menggunakanstandar gandadalam nilai kejahatanisraeldi gaza desak masyarakat dunia hukum israel atas jahat m


In [43]:
# Inisialisasi TF-IDF
vectorizer = TfidfVectorizer()

# Fit dan transform
X_tfidf = vectorizer.fit_transform(corpus)

print("Shape TF-IDF:", X_tfidf.shape)  # (jumlah dokumen, jumlah kata unik)


Shape TF-IDF: (961, 14937)


In [44]:
# Ubah ke DataFrame biar mudah dibaca
tfidf_data = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

# Simpan hasil ke CSV
tfidf_data.to_csv("/content/drive/MyDrive/Crawling/tfidf_hasilberita.csv", index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan 5 baris pertama
tfidf_data.head()


Data berhasil disimpan ke CSV!



,aalamiin,aanggota,aayush,ab,abad,abadi,abadirp,abai,abang,abar,...,zoom,zoomlion,zoudenbalch,zozo,zubimendi,zuhairi,zuhari,zulhas,zulkandar,zulkifli
0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.051355,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [45]:
# Tampilkan 10 kata dengan bobot TF-IDF tertinggi di dokumen pertama
print(tfidf_data.iloc[0].sort_values(ascending=False).head(10))

israel       0.348993
ktt          0.250200
hamas        0.218872
palestina    0.206748
gaza         0.201665
qatar        0.181047
al           0.161332
islam        0.161127
arab         0.150471
mohammed     0.144113
Name: 0, dtype: float64


## **CBOW**

In [ ]:
!pip install gensim


In [46]:
from gensim.models import Word2Vec

corpus = []
for col in data['stemmed']:
    word_list = col.split(" ")
    corpus.append(word_list)

# Tampilkan contoh dokumen pertama
print(corpus[0][:20])


['perdana', 'menteriqatarsheikh', 'mohammed', 'bin', 'abdulrahman', 'al', 'thani', 'desak', 'dunia', 'internasional', 'henti', 'menggunakanstandar', 'gandadalam', 'nilai', 'kejahatanisraeldi', 'gaza', 'desak', 'masyarakat', 'dunia', 'hukum']


In [47]:
# Training Word2Vec
model = Word2Vec(
    corpus,
    vector_size=56,   # ukuran embedding
    window=5,
    min_count=1,
    sg=0   # CBOW
)

In [48]:
import numpy as np

class MeanEmbeddingVectorizer:
    def __init__(self, model):
        self.model = model
        self.vector_size = model.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.array([
            np.mean([self.model.wv[word] for word in words if word in self.model.wv]
                    or [np.zeros(self.vector_size)], axis=0)
            for words in X
        ])

    def fit_transform(self, X, y=None):
        return self.fit(X).transform(X)


In [49]:
mean_embedding_vectorizer = MeanEmbeddingVectorizer(model)
mean_embedded = mean_embedding_vectorizer.fit_transform(corpus)

# simpan ke DataFrame
data['array'] = list(mean_embedded)

# cek panjang embedding
data['embedding_length'] = data['array'].str.len()
print(data[['stemmed','embedding_length']].head())


                                             stemmed  embedding_length
0  perdana menteriqatarsheikh mohammed bin abdulr...                56
1  sejarah pengambilalihan aset pemerintahamerika...                56
2  manchester cityberhasil membantaimanchester un...                56
3  danau besar inggris raya lough neagh sebut ala...                56
4  kaptenpersija jakarta rizky ridhomembuka dialo...                56


In [50]:
num_features = len(data['array'].iloc[0])  # asumsi semua list punya panjang sama
columns = [f'f{i+1}' for i in range(num_features)]

# Inisialisasi dictionary untuk menampung data per kolom
data_dict = {col: [] for col in columns}

# Looping setiap baris di kolom 'embedding'
for embedding_list in data['array']:
    for i, value in enumerate(embedding_list):
        data_dict[f'f{i+1}'].append(value)

# Buat DataFrame dari dictionary
embedding_data = pd.DataFrame(data_dict)

print(embedding_data)

           f1        f2        f3        f4        f5        f6        f7  \
0   -0.061124  0.388943  0.037789 -0.029327 -0.288770 -0.283967  0.190011   
1   -0.105286  0.390919 -0.077979  0.037513 -0.360901 -0.283402  0.146358   
2    0.105656  0.461451  0.140354 -0.128200 -0.486460 -0.335791  0.103388   
3   -0.098864  0.362757 -0.070238  0.046138 -0.319601 -0.256771  0.139617   
4   -0.029302  0.479108  0.054131 -0.081010 -0.397494 -0.372320  0.144724   
..        ...       ...       ...       ...       ...       ...       ...   
956 -0.087671  0.415406 -0.077237  0.074621 -0.355963 -0.291131  0.140551   
957 -0.031697  0.406228  0.070229 -0.043441 -0.263928 -0.328038  0.129155   
958 -0.331968  0.598429 -0.221568  0.026334 -0.462118 -0.456327  0.165128   
959 -0.049546  0.348308 -0.009140 -0.013331 -0.309740 -0.257473  0.153922   
960  0.135913  0.520326  0.158066 -0.165243 -0.552441 -0.382946  0.151454   

           f8        f9       f10  ...       f47       f48       f49  \
0  

In [51]:
embedding_data['isi_berita'] = data['isi_berita'].values

In [52]:
embedding_data

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f48,f49,f50,f51,f52,f53,f54,f55,f56,isi_berita
0,-0.061124,0.388943,0.037789,-0.029327,-0.288770,-0.283967,0.190011,-0.476891,-0.821618,-0.338692,...,0.462434,-0.310992,0.029424,-0.250927,0.708223,0.233474,0.168953,0.515069,-0.369724,Perdana MenteriQatarSheikh Mohammed bin Abdulr...
1,-0.105286,0.390919,-0.077979,0.037513,-0.360901,-0.283402,0.146358,-0.409859,-0.757152,-0.270631,...,0.484396,-0.311507,-0.006199,-0.175905,0.651922,0.262335,0.133534,0.483284,-0.331510,Sejarah pengambilalihan aset oleh pemerintahAm...
2,0.105656,0.461451,0.140354,-0.128200,-0.486460,-0.335791,0.103388,-0.260990,-0.896384,-0.504811,...,0.090727,-0.196515,0.265229,-0.191898,0.622192,0.144456,0.042511,0.401728,-0.325594,Manchester Cityberhasil membantaiManchester Un...
3,-0.098864,0.362757,-0.070238,0.046138,-0.319601,-0.256771,0.139617,-0.394236,-0.674381,-0.246394,...,0.455774,-0.270539,0.000312,-0.149505,0.620284,0.263066,0.155506,0.445676,-0.302075,"Danau terbesar diInggris Raya, Lough Neagh, di..."
4,-0.029302,0.479108,0.054131,-0.081010,-0.397494,-0.372320,0.144724,-0.422384,-0.918339,-0.436579,...,0.340769,-0.283895,0.155937,-0.241918,0.766189,0.206002,0.142269,0.484935,-0.380245,"KaptenPersija Jakarta,Rizky Ridhomembuka dialo..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
956,-0.087671,0.415406,-0.077237,0.074621,-0.355963,-0.291131,0.140551,-0.462373,-0.748476,-0.284967,...,0.479764,-0.340963,0.001288,-0.150226,0.656302,0.324024,0.179914,0.490846,-0.332932,Cara cekkesehatanparu-paru sering jadi pertany...
957,-0.031697,0.406228,0.070229,-0.043441,-0.263928,-0.328038,0.129155,-0.496561,-0.828501,-0.359899,...,0.408273,-0.326663,0.047887,-0.264965,0.664127,0.260698,0.190922,0.446294,-0.313926,Komisi III DPRsiap untuk mulai membahasRUU Per...
958,-0.331968,0.598429,-0.221568,0.026334,-0.462118,-0.456327,0.165128,-0.576548,-0.768186,-0.462838,...,0.834132,-0.534753,0.117812,-0.282475,1.001919,0.288449,0.295609,0.618062,-0.433497,Kementerian Keuangan (Kemenkeu) memastikan uan...
959,-0.049546,0.348308,-0.009140,-0.013331,-0.309740,-0.257473,0.153922,-0.354413,-0.698996,-0.272281,...,0.331755,-0.189537,0.068995,-0.154003,0.628650,0.207369,0.130908,0.409615,-0.312461,Gereja Setanmerupakan film horor terbaru Indon...


In [53]:
embedding_data.shape

(961, 57)

[Lanjut ke LDA](ektraksi_fitur_LDA.ipynb)
